# 🏅 KidStats 2035 — Youth Multi-Sport Analytics Dashboard

> **Year: 2035** | AI-Powered Youth Athlete Development Tracker
>
> *For the busy parent who has 5 minutes to check in on their kid's sports journey.*

---

### 🔧 Customize: Change the name, age, and seed below to generate your kid's dashboard

In [ ]:
# === CUSTOMIZE YOUR ATHLETE ===
ATHLETE_NAME = "Maya"     # Your kid's name
ATHLETE_AGE = 12           # Current age
RANDOM_SEED = 42           # Change for different data

# === Setup ===
import sys; sys.path.insert(0, 'src')
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from data_engine import *

athlete = generate_athlete(ATHLETE_NAME, ATHLETE_AGE, seed=RANDOM_SEED)
seasons = generate_season_data(athlete, seed=RANDOM_SEED)
radar = generate_skill_radar(athlete, seed=RANDOM_SEED)
risks = generate_injury_risk(athlete, seed=RANDOM_SEED)
growth = generate_growth_trajectory(athlete, seed=RANDOM_SEED)
schedule = generate_training_schedule(athlete, seed=RANDOM_SEED)

print(get_parent_summary(athlete, seasons))

## 📊 Performance Across Sports (2032-2035)

In [ ]:
df = pd.DataFrame([{
    'Sport': next(s['name'] for s in SPORTS if s['id'] == d.sport_id),
    'Year': d.year,
    'Performance': d.performance_score,
    'AI Predicted': d.ai_predicted_score,
    'Games': d.games_played,
    'Practice Hours': d.practice_hours,
} for d in seasons])

fig = px.line(df, x='Year', y='Performance', color='Sport',
              title=f"🏆 {athlete.name}'s Performance Trajectory",
              markers=True, template='plotly_dark')
fig.update_layout(yaxis_range=[0, 100], height=400,
                  font=dict(family='Georgia'), paper_bgcolor='#0a0a1a', plot_bgcolor='#12122a')
fig.show()

## 🎯 Skill Radar — Athletic Profile

In [ ]:
radar_dict = radar.to_dict()
categories = list(radar_dict.keys())
values = list(radar_dict.values())

fig = go.Figure(data=go.Scatterpolar(
    r=values + [values[0]],
    theta=categories + [categories[0]],
    fill='toself',
    fillcolor='rgba(107, 92, 231, 0.3)',
    line=dict(color='#e8a838', width=2),
    marker=dict(size=8, color='#ffd700'),
))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100], gridcolor='#2a2a4a'),
               angularaxis=dict(gridcolor='#2a2a4a'), bgcolor='#12122a'),
    title=f"🎯 {athlete.name}'s Skill Radar (Avg: {radar.average():.0f}/100)",
    template='plotly_dark', height=450,
    paper_bgcolor='#0a0a1a', font=dict(family='Georgia'),
)
fig.show()
print(f"Overall Athletic Score: {radar.average():.1f}/100")

## 🩹 Injury Risk Heatmap

In [ ]:
risk_df = pd.DataFrame([{'Zone': z.title(), 'Risk %': r * 100} for z, r in risks.items()])
risk_df = risk_df.sort_values('Risk %', ascending=True)

colors = ['#2ecc71' if r < 8 else '#f39c12' if r < 12 else '#e74c3c' for r in risk_df['Risk %']]

fig = go.Figure(go.Bar(
    x=risk_df['Risk %'], y=risk_df['Zone'], orientation='h',
    marker_color=colors, text=[f"{r:.1f}%" for r in risk_df['Risk %']],
    textposition='outside',
))
fig.update_layout(
    title=f"🩹 {athlete.name}'s Injury Risk by Zone",
    xaxis_title='Risk %', xaxis_range=[0, 25],
    template='plotly_dark', height=350,
    paper_bgcolor='#0a0a1a', plot_bgcolor='#12122a', font=dict(family='Georgia'),
)
fig.show()

## 📐 Growth Trajectory (Actual + AI Predicted)

In [ ]:
g_df = pd.DataFrame(growth)

fig = go.Figure()
actual = g_df[~g_df['predicted']]
predicted = g_df[g_df['predicted']]

fig.add_trace(go.Scatter(x=actual['year'], y=actual['height_cm'], name='Actual Height',
                         mode='lines+markers', line=dict(color='#6b5ce7', width=3),
                         marker=dict(size=10)))
fig.add_trace(go.Scatter(x=predicted['year'], y=predicted['height_cm'], name='Predicted Height',
                         mode='lines+markers', line=dict(color='#6b5ce7', width=2, dash='dash'),
                         marker=dict(size=8, symbol='diamond')))

fig.add_trace(go.Scatter(x=actual['year'], y=actual['weight_kg'], name='Actual Weight',
                         mode='lines+markers', line=dict(color='#e8a838', width=3),
                         marker=dict(size=10), yaxis='y2'))
fig.add_trace(go.Scatter(x=predicted['year'], y=predicted['weight_kg'], name='Predicted Weight',
                         mode='lines+markers', line=dict(color='#e8a838', width=2, dash='dash'),
                         marker=dict(size=8, symbol='diamond'), yaxis='y2'))

fig.update_layout(
    title=f"📐 {athlete.name}'s Growth Trajectory",
    yaxis=dict(title='Height (cm)', titlefont=dict(color='#6b5ce7'), tickfont=dict(color='#6b5ce7')),
    yaxis2=dict(title='Weight (kg)', titlefont=dict(color='#e8a838'), tickfont=dict(color='#e8a838'),
               overlaying='y', side='right'),
    template='plotly_dark', height=400,
    paper_bgcolor='#0a0a1a', plot_bgcolor='#12122a', font=dict(family='Georgia'),
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

## 🗓️ AI-Optimized Weekly Training Schedule

In [ ]:
s_df = pd.DataFrame(schedule)
intensity_colors = {'light': '#2ecc71', 'moderate': '#f39c12', 'high': '#e74c3c', 'none': '#555'}
s_df['color'] = s_df['intensity'].map(intensity_colors)

fig = go.Figure(go.Bar(
    x=s_df['day'], y=s_df['duration_min'],
    marker_color=s_df['color'],
    text=s_df['activity'], textposition='inside',
    hovertext=[f"{r['activity']}\n{r['duration_min']}min ({r['intensity']})" for _, r in s_df.iterrows()],
))

fig.update_layout(
    title=f"🗓️ {athlete.name}'s Optimal Weekly Schedule (AI-Generated)",
    yaxis_title='Duration (min)', template='plotly_dark', height=350,
    paper_bgcolor='#0a0a1a', plot_bgcolor='#12122a', font=dict(family='Georgia'),
)
fig.show()

total_min = sum(s['duration_min'] for s in schedule)
print(f"Total weekly training: {total_min} minutes ({total_min/60:.1f} hours)")

## 🤖 AI Coach Recommendation

In [ ]:
print(get_ai_recommendation(athlete, seasons))

---

*KidStats 2035 — Powered by AI Coach v12.3 | Youth Sports Analytics Platform*

*"Every child is an athlete. We just help them find their sport."*

---

💡 **Tip:** Change `ATHLETE_NAME`, `ATHLETE_AGE`, and `RANDOM_SEED` at the top to generate a fresh dashboard!